# VZP: реальные приглашения на маммографический скрининг

Одна строка — приглашение, а не независимая клиентка. Описание выгрузки ограничивает
когорту непрерывным страхованием VZP в 2015–2025 годах. Переносимость на другие группы
не проверена. Этот ноутбук исследует ассоциации и планирует эксперимент; он не обучает модель.

Запуск: установить `pip install -e ".[dev]"`, положить CSV в путь из конфигурации и выполнить
все ячейки. Для другого локального файла задать `SCREENING_EDA_DATA`.
Все таблицы и PNG сохраняются в игнорируемом `reports/`; ноутбук в Git хранится без outputs.
Не сохранять и не коммитить строки с идентификаторами.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from screening.config import load_config
from screening.evaluation.real_data import (
    build_tables,
    make_figures,
    read_delivery,
    save_tables,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
cfg = load_config(ROOT / 'configs/config.yaml')
source = Path(os.environ.get('SCREENING_EDA_DATA', cfg['data']['path']))
if not source.is_absolute():
    source = ROOT / source
if not source.is_file():
    raise FileNotFoundError('Provide the VZP CSV locally; raw data must never be committed.')
report_dir = ROOT / 'reports/eda'
figure_dir = ROOT / 'reports/figures'


## Проверки и единица наблюдения

Флаги `SCREENING_3M/6M/12M_FLAG` — исходы после приглашения, не входы модели.
Для EDA используем явку (`A = 1`), модель предсказывает противоположный исход — неявку.
Даты читаются строго как `%d.%m.%Y`. Проверяем идентификаторы, уникальность клиентка/год
и вложенность накопленных исходов: 3M ≤ 6M ≤ 12M.

Закрытие 365-дневного окна должно подтверждаться полнотой регистрации услуг;
похожая явка по годам сама по себе не доказывает полноту наблюдения.


In [ ]:
df = read_delivery(source)
display(pd.DataFrame([{
    'invitations': len(df), 'clients': df.PERSON_ID.nunique(),
    'date_min': df.date.min(), 'date_max': df.date.max(),
    'attendance_365d': df.attended_12m.mean(),
    'first_flag_counter_disagreements': int((df.FIRST_INVITATION.eq('A') != df.PRIOR_INVITATION_COUNT.eq(1)).sum()),
}]))
display(df.isna().sum().rename('missing').to_frame())


## Единые воспроизводимые таблицы

Ставки за все годы взвешены по приглашениям; 95% интервалы учитывают повторные записи
одной клиентки через кластерную дисперсию. Для годовых групп с одной записью на клиентку
используются интервалы Wilson. Это точечные, не одновременные интервалы.

Список территорий для гипотезы фиксируется по 2020–2022 годам (`eda` в конфиге),
затем применяется к 2024 году. Рейтинг 2024 года не используется для выбора этих территорий.


In [ ]:
tables = build_tables(df, cfg)
save_tables(tables, report_dir, source, cfg)
figures = make_figures(tables, figure_dir, cfg['split']['test_year'])
display(tables['by_year'])
print(f'Saved {len(tables)} aggregate tables and {len(figures)} PNG figures.')


## Приглашения, возраст и взаимодействие с медициной

Счётчик приглашений показан в исходном виде, включая текущее приглашение.
Для подсчёта первых приглашений отдельно используем переданный `FIRST_INVITATION`.
Счётчики процедур не делятся на два: соответствие кодов отдельным посещениям не подтверждено.
Возраст, история приглашений и профилактика связаны между собой; это не причинные эффекты.


In [ ]:
display(tables['first_invitation'])
for name in ['invitations', 'age', 'contacts', 'gyn_prevention', 'other_screening']:
    display(Image(filename=str(figure_dir / f'{name}.png'), width=900))


## 75 территорий VZP

Территориальное отделение VZP не равно точному городу проживания или измеренной доступности.
Рейтинг показывает грубую явку без поправки на возраст и другие различия состава населения.
Перекрывающиеся интервалы и 75 сравнений не позволяют объявлять отдельные места причинным
«географическим барьером». Полная таблица, включая `n`, хранится в CSV.


In [ ]:
display(tables['regions_reporting_year'])
display(tables['regional_reference'].loc[lambda x: x.selected])
display(Image(filename=str(figure_dir / 'regions_ranking.png'), width=950))


## Скорость реакции: 90 / 180 / 365 дней

Флаги накопленные. Разность явки 6M−3M показывает долю приглашений с первым учтённым
скринингом между этими контрольными точками; 12M−6M — после 180-го дня до 365-го.
Точной даты обследования и оптимального дня напоминания здесь нет.
Напоминание, например через 90 дней, остаётся гипотезой для рандомизированной проверки.


In [ ]:
timing = tables['response_timing'].copy()
timing['additional_attendance_since_previous_horizon'] = timing.attendance.diff().fillna(timing.attendance)
display(timing)
display(Image(filename=str(figure_dir / 'response_timing.png'), width=900))


## Приглашения вскоре после зарегистрированного скрининга

Считаем отдельно окна 0–1 и 0–6 месяцев. Окна вложены: их нельзя складывать.
Это кандидаты на проверку процесса рассылки, а не доказанно ненужные приглашения.
Месяцы могут быть округлены; ноль сам по себе не доказывает ошибку даты.

`eda.letter_cost_czk` — явно условная стоимость одного письма. Произведение числа писем
на эту цену — сценарная валовая стоимость, не подтверждённая экономия. Отмену приглашений
этот анализ не обосновывает; нужны проверка записей и согласование владельцем программы.


In [ ]:
display(tables['recent_screening_cost'])
display(Image(filename=str(figure_dir / 'recent_screening_cost.png'), width=1000))


## Гипотезы сегментов и дизайн эксперимента

Таблицы гипотез пересекаются и не суммируются. Для A/B используем отдельные непересекающиеся
группы 2024 года с приоритетом текущего production-кода. Плановая группа `regional` добавляется
только среди оставшегося `other`; production-правила этим ноутбуком не изменяются.
Группа недавно прошедших скрининг не включается в план усиления приглашений.

Размеры групп рассчитаны через `experiment.power.n_per_group`: независимые клиентки,
два плеча 1:1, двусторонняя alpha=0.05, power=0.80, абсолютные MDE из конфигурации.
Это отдельные варианты A/B, не готовый многогрупповой тест с общим контролем.
Численность 2024 года — верхний ориентир годового набора, не обещание доступного набора сейчас.


In [ ]:
display(tables['hypotheses_all_years'])
display(tables['hypotheses_reporting_year'])
display(tables['segments_exclusive'])
display(tables['experiment_sizes'])
assert tables['segments_exclusive'].n.sum() == df.year.eq(cfg['split']['test_year']).sum()
assert tables['segments_exclusive'].n.eq(tables['segments_exclusive'].clients).all()


## Что ещё потребуется перед A/B

- Зафиксировать интервенцию, критерии участия и одну запись на клиентку в эксперименте.
- Использовать реальные прогнозы замороженной модели для квинтилей риска; этот EDA их не выдумывает.
- Рандомизировать 1:1 внутри сегмента × квинтиля, сохранить назначения и seed.
- Основной исход — явка за 365 дней; 90 дней — вторичный ранний сигнал, а не замена основного.
- Анализ ITT: не исключать недозвоны и недоставленные сообщения. Неизвестный исход не равен нулю.
- Один финальный анализ; ранние остановки по безопасности/качеству данных, без ежедневного поиска p<0.05.
- Перед несколькими подтверждающими сравнениями пересчитать размер с учётом множественности.

Подробности и фактические таблицы: `docs/segments-real-data.md`, `docs/experiment-design.md`.
